# 01 - Data Cleaning & Profiling

**Objective:** Load raw datasets, profile data quality issues, and produce cleaned outputs.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add parent directory to path
sys.path.insert(0, os.path.abspath('..'))

from src.ingest import ZomatoDataIngestor
from src.clean import DataCleaner
from src.utils import setup_logger

# Setup logging
logger = setup_logger(__name__)

print("✅ All imports successful!")

✅ All imports successful!


## 1. Load All Datasets

In [2]:
# Initialize ingestor
ingestor = ZomatoDataIngestor(raw_data_path='../data/raw/', logger=logger)

# Load all datasets
datasets = ingestor.load_all_datasets()

print(f"\n✅ Loaded {len(datasets)} datasets")
for name in datasets.keys():
    print(f"  - {name}: {len(datasets[name]):,} rows")

2026-08-08 14:39:17 - __main__ - INFO - 
2026-08-08 14:39:17 - __main__ - INFO - LOADING ALL DATASETS
2026-08-08 14:39:17 - __main__ - INFO - ======================================================================
2026-08-08 14:39:17 - __main__ - WARNING - ⚠ File not found: ..\data\raw\customers.csv
2026-08-08 14:39:17 - __main__ - INFO - ✓ Loaded ..\data\raw\restaurants.csv (1,200 rows)
2026-08-08 14:39:17 - __main__ - INFO - ✓ Loaded ..\data\raw\orders.csv (20,705 rows)
2026-08-08 14:39:17 - __main__ - INFO - ✓ Loaded ..\data\raw\order_items.csv (42,378 rows)
2026-08-08 14:39:17 - __main__ - INFO - ✓ Loaded ..\data\raw\menu.csv (8,997 rows)
2026-08-08 14:39:17 - __main__ - INFO - ✓ Loaded ..\data\raw\delivery_partners.csv (2,000 rows)
2026-08-08 14:39:17 - __main__ - WARNING - ⚠ File not found: ..\data\raw\customer_feedback.csv
2026-08-08 14:39:17 - __main__ - WARNING - ⚠ File not found: ..\data\raw\payments.csv
2026-08-08 14:39:17 - __main__ - INFO - ✓ Loaded ..\data\raw\promotions.c


✅ Loaded 8 datasets
  - restaurants: 1,200 rows
  - orders: 20,705 rows
  - order_items: 42,378 rows
  - menu: 8,997 rows
  - delivery_partners: 2,000 rows
  - promotions: 300 rows
  - weather: 18,264 rows
  - traffic: 18,335 rows


## 2. Profile Data Quality Issues

In [3]:
# Profile all datasets
quality_reports = ingestor.profile_all_datasets()

# FIXED: check_dataset_integrity (singular, not plural)
integrity_issues = ingestor.check_dataset_integrity()

# FIXED: summary (not sumaary - typo fixed)
summary = ingestor.get_dataset_summary()
print("\n" + summary.to_string())

2026-08-08 14:39:22 - __main__ - INFO - 
2026-08-08 14:39:22 - __main__ - INFO - PROFILING ALL DATASETS
2026-08-08 14:39:22 - __main__ - INFO - ======================================================================
2026-08-08 14:39:22 - __main__ - INFO - 
2026-08-08 14:39:22 - __main__ - INFO - DATA QUALITY REPORT: restaurants
2026-08-08 14:39:22 - __main__ - INFO - ============================================================
2026-08-08 14:39:22 - __main__ - INFO - Rows: 1,200
2026-08-08 14:39:22 - __main__ - INFO - Columns: 13
2026-08-08 14:39:22 - __main__ - INFO - Duplicates: 0
2026-08-08 14:39:22 - __main__ - INFO - Memory: 0.65 MB
2026-08-08 14:39:22 - __main__ - INFO - Missing Values:
restaurantid       0
restaurantname     0
cuisine           12
city               0
area               0
openingtime        0
closingtime        0
rating            22
averagecost        0
ownername          0
restauranttype     0
latitude           0
longitude          0
dtype: int64
2026-08-08 14:


             Dataset   Rows  Columns  Memory (MB)  Missing %
0        restaurants   1200       13     0.646235   0.217949
1             orders  20705       15     7.695377   4.426628
2        order_items  42378        6     1.940033   0.171079
3               menu   8997        8     1.999284   0.183394
4  delivery_partners   2000       10     0.696497   0.170000
5         promotions    300        6     0.082885   0.000000
6            weather  18264        7     3.925541   0.281584
7            traffic  18335        6     4.745540   0.133624


## 3. Clean Datasets

In [5]:
# FIXED: DataCleaner (capital C, not Datacleaner)
cleaner = DataCleaner(logger=logger)

# Clean each dataset
cleaned_datasets = {}

print("\n🧹 Cleaning datasets...")
cleaned_datasets['customers'] = cleaner.clean_customers(datasets['customers'])
cleaned_datasets['restaurants'] = cleaner.clean_restaurants(datasets['restaurants'])
cleaned_datasets['orders'] = cleaner.clean_orders(datasets['orders'])
cleaned_datasets['menu'] = cleaner.clean_menu(datasets['menu'])
cleaned_datasets['delivery_partners'] = cleaner.clean_delivery_partners(datasets['delivery_partners'])
cleaned_datasets['customer_feedback'] = cleaner.clean_customer_feedback(datasets['customer_feedback'])
cleaned_datasets['payments'] = cleaner.clean_payments(datasets['payments'])

print("\n✅ Core datasets cleaned!")

# FIXED: order_items (underscore, not hyphen)
for table in ['order_items', 'promotions', 'cities', 'weather', 'traffic']:
    if table in datasets:
        cleaned_datasets[table] = datasets[table].copy()
        print(f"  - {table}: {len(cleaned_datasets[table]):,} rows")


🧹 Cleaning datasets...


KeyError: 'customers'

## 4. Export Cleaned Datasets

In [6]:
# Export cleaned CSV files
import os 
os.makedirs('../data/cleaned/', exist_ok=True)

print("\n📁 Exporting cleaned datasets...")
for table_name, df in cleaned_datasets.items():
    filepath = f'../data/cleaned/{table_name}_cleaned.csv'
    df.to_csv(filepath, index=False, encoding='utf-8')
    print(f"  ✓ Exported {table_name}: {len(df):,} rows")

print("\n✅ All datasets exported successfully!")


📁 Exporting cleaned datasets...

✅ All datasets exported successfully!


##  5. Generate Cleaning Log

In [ ]:
# Export cleaning summary
cleaning_summary = cleaner.get_cleaning_summary()
print("\n📊 Data Cleaning Summary:")
print("\n" + cleaning_summary.to_string())

# Save to CSV
cleaner.export_cleaning_log('../data/cleaned/DATA_CLEANING_LOG.csv')
print("\n✅ Cleaning log exported!")